# 03 Machine Learning + Deep Learning

Merged notebook for modules 03a-04f in tutorial order.

In [ ]:
# @title ⚙️ Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib scikit-learn torch -q
print("✅ Packages installed")

In [ ]:
# @title 📂 Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("✅ Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("✅ Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("✅ Data loaded successfully")


## 03a - Model Training Validation

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_ml_conv_1phase
from tutorial_utils.sections import ml_conv_1phase as s03a


def create_conv_ml_1phase_demo(
    synth_samples_per_formula=12,
    random_seed=42,
    output_dir="outputs/ml/conv",
    show_steps=True,
):
    """Conventional ML (k-NN / RF / SVM) for one-phase identification."""
    return run_ml_conv_1phase(
        synth_samples_per_formula=synth_samples_per_formula,
        random_seed=random_seed,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_conv_ml_1phase_profile(phase="TiO2", seed=42, show_plot=True):
    """Expose synthetic-profile generation used by the 1-phase ML section."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03a.MIN_ANGLE, s03a.MAX_ANGLE, s03a.NUM_POINTS)
    refs = s03a.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    candidates = [r for r in refs if r["formula"] == phase]
    if not candidates:
        raise ValueError(f"Phase '{phase}' not found in reference library")

    ref = candidates[0]
    synthetic = s03a.simulate_artifact_profile(grid, ref["peak_pos"], ref["peak_int"], rng)

    exp_path = Path(f"data/exp_patterns/one_phase/{phase}.xy")
    experimental = s03a.preprocess_experimental_pattern(exp_path, grid) if exp_path.exists() else None

    print(f"Step 1: built two-theta grid with {len(grid)} points")
    print(f"Step 2: loaded sticks for '{ref['phase']}'")
    print("Step 3: generated one artifact-rich synthetic profile")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8, label="Synthetic")
        if experimental is not None:
            plt.plot(grid, experimental, color="black", linewidth=1.6, alpha=0.9, label="Experimental")
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title(f"1-phase ML inner step: profile generation ({phase})")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "experimental": experimental, "reference_phase": ref["phase"]}


# 03a — Conventional ML: Training & Validation

We train k-NN, Random Forest, and SVM classifiers on synthetic single-phase profiles and test on experimental data.

## Runtime Note
For a live tutorial, we use a smaller synthetic dataset. Increase `SYNTH_SAMPLES_PER_FORMULA` for higher-fidelity training offline.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Conventional ML (1-Phase)
create_conv_ml_1phase_demo()

# Try on your own:
# create_conv_ml_1phase_demo(synth_samples_per_formula=24)
# inspect_conv_ml_1phase_profile(phase="Li2MnO3", seed=7)


## What To Observe
Compare validation vs experimental-test accuracy across the three model families.

In [ ]:
display(Image("outputs/ml/conv/model_accuracy_summary.png"))

## Summary
- Synthetic augmentation enables supervised learning with limited experimental labels.
- Different model families show different generalization behavior.
- Validation and test gaps highlight domain-shift effects.

## Next Steps
Continue to the next section below in this notebook.

## 03b - Multiphase ML

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_ml_multiphase
from tutorial_utils.sections import ml_multiphase as s03b


def create_conv_ml_multiphase_demo(
    synth_samples_per_formula=10,
    random_seed=42,
    single_phase_fraction=0.15,
    output_dir="outputs/ml/multiphase",
    show_steps=True,
):
    """Conventional ML multiphase demo with thresholded multi-label outputs."""
    return run_ml_multiphase(
        synth_samples_per_formula=synth_samples_per_formula,
        random_seed=random_seed,
        single_phase_fraction=single_phase_fraction,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_conv_ml_multiphase_profile(formulas=("TiO2", "ZrO2"), seed=42, show_plot=True):
    """Expose multiphase synthetic-profile generation used in conventional ML."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03b.MIN_ANGLE, s03b.MAX_ANGLE, s03b.NUM_POINTS)
    refs_by_formula = s03b.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    formulas = tuple(formulas)
    missing = [f for f in formulas if f not in refs_by_formula]
    if missing:
        raise ValueError(f"Unknown formulas: {missing}")

    synthetic = s03b.simulate_multiphase_profile(grid, list(formulas), refs_by_formula, rng)
    print(f"Step 1: loaded {len(refs_by_formula)} phase families")
    print(f"Step 2: sampled components for formulas: {formulas}")
    print("Step 3: mixed components + background/noise into one profile")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title(f"Multiphase ML inner step: synthetic mixture {formulas}")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "formulas": formulas}


def inspect_thresholding(raw_scores=(0.12, 0.44, 0.73), threshold=0.5):
    """Show how the multiphase thresholding logic converts scores to binary labels."""
    scores = np.asarray(raw_scores, dtype=float)[None, :]
    pred_bin = s03b.predict_with_threshold(scores, threshold=threshold)
    print("Raw scores:", scores.ravel())
    print(f"Threshold: {threshold}")
    print("Predicted binary labels:", pred_bin.ravel())
    return pred_bin.ravel()


# 03b — Conventional ML: Multiphase Classification

This notebook extends to multi-label phase prediction using synthetic mixtures and thresholded outputs.

## Runtime Note
The dataset size is reduced for in-session runtime. Increase sample counts for stronger offline models.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Conventional ML (Multiphase)
create_conv_ml_multiphase_demo()

# Try on your own:
# create_conv_ml_multiphase_demo(synth_samples_per_formula=20, single_phase_fraction=0.25)
# inspect_conv_ml_multiphase_profile(formulas=("Li2CO3", "TiO2", "ZrO2"), seed=3)
# inspect_thresholding(raw_scores=(0.2, 0.49, 0.51), threshold=0.5)


## What To Observe
Focus on precision/recall/F1 tradeoffs after threshold selection.

In [ ]:
display(Image("outputs/ml/multiphase/model_test-metric_summary.png"))

## Summary
- Multiphase ID is naturally a multi-label problem.
- Threshold tuning materially changes precision-recall behavior.
- Synthetic mixture design controls model robustness.

## Next Steps
Continue to the next section below in this notebook.

## 04a - NNs 1phase

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_nn_1phase
from tutorial_utils.sections import dl_nn_1phase as s03c


def create_nn_1phase_demo(
    synth_samples_per_formula=12,
    nn_max_iter=80,
    random_seed=42,
    output_dir="outputs/dl/nn_1phase",
    show_steps=True,
):
    """Feed-forward neural network demo for one-phase prediction."""
    return run_nn_1phase(
        synth_samples_per_formula=synth_samples_per_formula,
        nn_max_iter=nn_max_iter,
        random_seed=random_seed,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_nn_1phase_profile(phase="TiO2", seed=42, show_plot=True):
    """Expose synthetic profile generation for the 1-phase NN section."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03c.MIN_ANGLE, s03c.MAX_ANGLE, s03c.NUM_POINTS)
    refs = s03c.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    candidates = [r for r in refs if r["formula"] == phase]
    if not candidates:
        raise ValueError(f"Phase '{phase}' not found in reference library")

    ref = candidates[0]
    synthetic = s03c.simulate_artifact_profile(grid, ref["peak_pos"], ref["peak_int"], rng)

    print(f"Step 1: grid points = {len(grid)}")
    print(f"Step 2: sampled reference stick pattern = {ref['phase']}")
    print("Step 3: created artifact-rich synthetic profile for NN training")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title(f"NN 1-phase inner step: synthetic profile ({phase})")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "reference_phase": ref["phase"]}


# 04a — Neural Networks: 1-Phase

A feedforward neural network is trained on synthetic 1-phase profiles and evaluated on experimental patterns.

## Runtime Note
`NN_MAX_ITER` is reduced here for live speed. Increase it for full convergence offline.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run NN (1-Phase)
create_nn_1phase_demo()

# Try on your own:
# create_nn_1phase_demo(nn_max_iter=140)
# inspect_nn_1phase_profile(phase="MnO2", seed=11)


## What To Observe
Inspect the training loss trend and compare validation vs test accuracy.

In [ ]:
display(Image("outputs/dl/nn_1phase/nn_loss_curve.png"))
display(Image("outputs/dl/nn_1phase/nn_accuracy_summary.png"))

## Summary
- Simple dense NNs can capture nonlinear XRD-feature interactions.
- Training duration strongly affects final accuracy.
- Synthetic realism still matters as much as network choice.

## Next Steps
Continue to the next section below in this notebook.

## 04b - NNs Multiphase

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_nn_multiphase
from tutorial_utils.sections import dl_nn_multiphase as s03d


def create_nn_multiphase_demo(
    synth_samples_per_formula=8,
    nn_max_iter=80,
    random_seed=42,
    prediction_threshold=0.50,
    output_dir="outputs/dl/nn_multiphase",
    show_steps=True,
):
    """Feed-forward neural network demo for multiphase prediction."""
    return run_nn_multiphase(
        synth_samples_per_formula=synth_samples_per_formula,
        nn_max_iter=nn_max_iter,
        random_seed=random_seed,
        prediction_threshold=prediction_threshold,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_nn_multiphase_profile(formulas=("TiO2", "ZrO2"), seed=42, show_plot=True):
    """Expose multiphase synthetic-profile generation for NN training."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03d.MIN_ANGLE, s03d.MAX_ANGLE, s03d.NUM_POINTS)
    refs_by_formula = s03d.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    formulas = tuple(formulas)
    missing = [f for f in formulas if f not in refs_by_formula]
    if missing:
        raise ValueError(f"Unknown formulas: {missing}")

    synthetic = s03d.simulate_multiphase_profile(grid, list(formulas), refs_by_formula, rng)
    print(f"Step 1: built grid with {len(grid)} points")
    print(f"Step 2: selected components {formulas}")
    print("Step 3: generated one synthetic multiphase profile")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title(f"NN multiphase inner step: synthetic mixture {formulas}")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "formulas": formulas}


def inspect_nn_thresholding(raw_scores=(0.15, 0.55, 0.80), threshold=0.50):
    scores = np.asarray(raw_scores, dtype=float)[None, :]
    pred_bin = s03d.predict_binary_labels(scores, threshold=threshold)
    print("Raw scores:", scores.ravel())
    print(f"Threshold: {threshold}")
    print("Predicted binary labels:", pred_bin.ravel())
    return pred_bin.ravel()


# 04b — Neural Networks: Multiphase

We apply a dense multi-label NN to mixed-phase data and evaluate micro-averaged metrics.

## Runtime Note
This run uses reduced synthetic data and iterations to keep in-class runtime manageable.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run NN (Multiphase)
create_nn_multiphase_demo()

# Try on your own:
# create_nn_multiphase_demo(prediction_threshold=0.40)
# inspect_nn_multiphase_profile(formulas=("Li2CO3", "TiO2", "ZrO2"), seed=9)
# inspect_nn_thresholding(raw_scores=(0.1, 0.48, 0.92), threshold=0.5)


## What To Observe
Check whether recall and precision remain balanced at the fixed threshold.

In [ ]:
display(Image("outputs/dl/nn_multiphase/nn_loss_curve.png"))
display(Image("outputs/dl/nn_multiphase/nn_test-metric_summary.png"))

## Summary
- Dense NNs can perform multi-label phase inference.
- Threshold choice is critical for practical decoding.
- Model capacity and training data diversity must be balanced.

## Next Steps
Continue to the next section below in this notebook.

## 04c - CNNs Multiphase

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_cnn_multiphase
from tutorial_utils.sections import dl_cnn_multiphase as s03e


def create_cnn_multiphase_demo(
    synth_samples_per_formula=8,
    nn_max_iter=20,
    random_seed=42,
    output_dir="outputs/dl/cnn_multiphase",
    show_steps=True,
):
    """CNN multiphase demo with synthetic augmentation."""
    return run_cnn_multiphase(
        synth_samples_per_formula=synth_samples_per_formula,
        nn_max_iter=nn_max_iter,
        random_seed=random_seed,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_cnn_multiphase_profile(formulas=("TiO2", "ZrO2"), seed=42, show_plot=True):
    """Expose augmented synthetic-profile generation for CNN multiphase training."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03e.MIN_ANGLE, s03e.MAX_ANGLE, s03e.NUM_POINTS)
    refs_by_formula = s03e.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    synthetic = s03e.simulate_multiphase_profile(grid, list(formulas), refs_by_formula, rng)
    print(f"Shift range: {s03e.UNIFORM_SHIFT_RANGE}, displacement range: {s03e.SAMPLE_DISPLACEMENT_RANGE_MM}")
    print(f"Generated profile with {len(grid)} points for formulas: {tuple(formulas)}")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title("CNN multiphase inner step: augmented synthetic profile")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "formulas": tuple(formulas)}


# 04c — CNNs: Multiphase

A 1D CNN is trained on synthetic mixtures to capture local peak-shape and neighborhood features.

## Option A / Option B
Option A: run short training (shown below).

Option B: if you provide `data/pretrained/cnn_multiphase.pt`, load it in your own inference workflow.

In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_multiphase.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run CNN (Multiphase)
create_cnn_multiphase_demo()

# Try on your own:
# create_cnn_multiphase_demo(synth_samples_per_formula=14, nn_max_iter=40)
# inspect_cnn_multiphase_profile(formulas=("MnO2", "TiO2"), seed=5)


## What To Observe
Compare the CNN metric summary against the dense-NN multiphase baseline.

In [ ]:
display(Image("outputs/dl/cnn_multiphase/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_multiphase/nn_test-metric_summary.png"))

## Summary
- 1D CNNs encode local line-shape context directly.
- Short demo training is useful for workflow illustration, not final performance.
- Checkpoints are recommended for live sessions with tight time budgets.

## Next Steps
Continue to the next section below in this notebook.

## 04d - No Augmentation

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_cnn_no_augmentation
from tutorial_utils.sections import dl_cnn_no_augmentation as s03f


def create_cnn_no_augmentation_demo(
    synth_samples_per_formula=8,
    nn_max_iter=20,
    random_seed=42,
    output_dir="outputs/dl/cnn_no_augmentation",
    show_steps=True,
):
    """CNN ablation demo with minimal augmentation."""
    return run_cnn_no_augmentation(
        synth_samples_per_formula=synth_samples_per_formula,
        nn_max_iter=nn_max_iter,
        random_seed=random_seed,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_cnn_no_augmentation_profile(formulas=("TiO2", "ZrO2"), seed=42, show_plot=True):
    """Expose near-ideal synthetic profile generation (no heavy augmentation)."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03f.MIN_ANGLE, s03f.MAX_ANGLE, s03f.NUM_POINTS)
    refs_by_formula = s03f.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    synthetic = s03f.simulate_multiphase_profile(grid, list(formulas), refs_by_formula, rng)
    print(f"Ideal FWHM={s03f.IDEAL_FWHM}, ideal eta={s03f.IDEAL_ETA}")
    print(f"Generated profile with {len(grid)} points for formulas: {tuple(formulas)}")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title("CNN no-augmentation inner step: synthetic profile")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "formulas": tuple(formulas)}


# 04d — Ablation: No Augmentation

This ablation trains a CNN on idealized synthetic data without the broader artifact model.

## Option A / Option B
Option A: run short no-augmentation training.

Option B: use a saved checkpoint (if provided) from `data/pretrained/`.

In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_no_augmentation.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run CNN (No Augmentation)
create_cnn_no_augmentation_demo()

# Try on your own:
# create_cnn_no_augmentation_demo(nn_max_iter=40)
# inspect_cnn_no_augmentation_profile(formulas=("Li2CO3", "TiO2"), seed=5)


## What To Observe
Use this as a baseline to compare against artifact-aware augmentation variants.

In [ ]:
display(Image("outputs/dl/cnn_no_augmentation/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_no_augmentation/nn_test-metric_summary.png"))

## Summary
- No-augmentation training is a useful controlled baseline.
- Generalization typically drops when synthetic variability is too narrow.
- Ablations help quantify which augmentations matter most.

## Next Steps
Continue to the next section below in this notebook.

## 04e - Random Shifts

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_cnn_random_shifts
from tutorial_utils.sections import dl_cnn_random_shifts as s03g


def create_cnn_random_shifts_demo(
    synth_samples_per_formula=8,
    nn_max_iter=20,
    random_seed=42,
    output_dir="outputs/dl/cnn_random_shifts",
    show_steps=True,
):
    """CNN ablation demo where augmentation is mainly random peak shifts."""
    return run_cnn_random_shifts(
        synth_samples_per_formula=synth_samples_per_formula,
        nn_max_iter=nn_max_iter,
        random_seed=random_seed,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_cnn_random_shift_profile(formulas=("TiO2", "ZrO2"), seed=42, show_plot=True):
    """Expose shift-only augmentation used in this ablation."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03g.MIN_ANGLE, s03g.MAX_ANGLE, s03g.NUM_POINTS)
    refs_by_formula = s03g.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    synthetic = s03g.simulate_multiphase_profile(grid, list(formulas), refs_by_formula, rng)
    print(f"Max peak-shift magnitude: {s03g.MAX_PEAK_SHIFT_MAGNITUDE}")
    print(f"Generated profile with {len(grid)} points for formulas: {tuple(formulas)}")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title("CNN random-shifts inner step: synthetic profile")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "formulas": tuple(formulas)}


# 04e — Ablation: Random Shifts

This variant keeps idealized patterns but injects random peak-position shifts as a lightweight augmentation.

## Option A / Option B
Option A: run short random-shift training.

Option B: use a saved checkpoint from `data/pretrained/` if you have one.

In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_random_shifts.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run CNN (Random Shifts)
create_cnn_random_shifts_demo()

# Try on your own:
# create_cnn_random_shifts_demo(synth_samples_per_formula=14)
# inspect_cnn_random_shift_profile(formulas=("Li2CO3", "TiO2", "ZrO2"), seed=6)


## What To Observe
Compare this model against both no-augmentation and full-artifact CNN training.

In [ ]:
display(Image("outputs/dl/cnn_random_shifts/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_random_shifts/nn_test-metric_summary.png"))

## Summary
- Random shifts target one important nuisance factor: peak-position drift.
- Single-factor augmentation usually helps less than full artifact simulation.
- Ablation studies clarify augmentation ROI.

## Next Steps
Continue to the next section below in this notebook.

## 04f - Mixture of Experts

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from tutorial_utils.runners import run_mixture_of_experts
from tutorial_utils.sections import dl_moe as s03h


def create_moe_demo(
    synth_samples_per_formula=6,
    nn_max_iter=20,
    min_epochs=5,
    patience=4,
    random_seed=42,
    output_dir="outputs/dl/mixture_of_experts",
    show_steps=True,
):
    """Mixture-of-experts multiphase demo."""
    return run_mixture_of_experts(
        synth_samples_per_formula=synth_samples_per_formula,
        nn_max_iter=nn_max_iter,
        min_epochs=min_epochs,
        patience=patience,
        random_seed=random_seed,
        output_dir=output_dir,
        show_steps=show_steps,
    )


def inspect_moe_profile(formulas=("TiO2", "ZrO2"), seed=42, show_plot=True):
    """Expose synthetic-profile generation used before expert training."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(s03h.MIN_ANGLE, s03h.MAX_ANGLE, s03h.NUM_POINTS)
    refs_by_formula = s03h.load_reference_sticks(sorted(Path("data/reference_structures").glob("*.cif")))

    synthetic = s03h.simulate_multiphase_profile(grid, list(formulas), refs_by_formula, rng)
    print(f"Experts use threshold={s03h.PREDICTION_THRESHOLD}, max epochs={s03h.NN_MAX_ITER}")
    print(f"Generated profile with {len(grid)} points for formulas: {tuple(formulas)}")

    if show_plot:
        plt.figure(figsize=(8, 3.8))
        plt.plot(grid, synthetic, color="#1f4ed8", linewidth=1.8)
        plt.xlabel("2θ")
        plt.ylabel("Normalized intensity")
        plt.title("Mixture-of-experts inner step: synthetic profile")
        plt.tight_layout()
        plt.show()

    return {"two_theta": grid, "synthetic": synthetic, "formulas": tuple(formulas)}


def inspect_moe_thresholding(raw_scores=(0.05, 0.51, 0.90), threshold=0.50):
    scores = np.asarray(raw_scores, dtype=float)[None, :]
    pred_bin = s03h.predict_binary_labels(scores, threshold=threshold)
    print("Raw expert scores:", scores.ravel())
    print(f"Threshold: {threshold}")
    print("Predicted binary labels:", pred_bin.ravel())
    return pred_bin.ravel()


# 04f — Mixture of Experts

One lightweight binary expert is trained per phase label, then combined for final multi-label prediction.

## Option A / Option B
Option A: run a short MoE training demo.

Option B: load per-expert checkpoints from `data/pretrained/` in a custom workflow.

In [ ]:
PRETRAINED_PATH = "data/pretrained/moe_experts"
print("Found pretrained expert folder:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Mixture-of-Experts
create_moe_demo()

# Try on your own:
# create_moe_demo(nn_max_iter=40, patience=6)
# inspect_moe_profile(formulas=("Li2CO3", "TiO2", "ZrO2"), seed=6)
# inspect_moe_thresholding(raw_scores=(0.1, 0.4, 0.95), threshold=0.5)


## What To Observe
MoE can improve flexibility by allowing each phase detector to specialize.

In [ ]:
display(Image("outputs/dl/mixture_of_experts/nn_loss_curve.png"))
display(Image("outputs/dl/mixture_of_experts/nn_test-metric_summary.png"))

## Summary
- MoE decomposes multi-label classification into expert subproblems.
- Early stopping helps keep many experts computationally manageable.
- Expert-level metrics can reveal label-specific weaknesses.

## Next Steps
Continue to **04 — Open Challenge**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/04_Challenge.ipynb)